In [1]:
import pickle
import numpy as np

In [3]:
pipe = pickle.load(open('pipe.pkl','rb'))

---
## Inference With Pipeline — Same Input, Zero Manual Work

This notebook performs the same inference task as `WithoutPipeline2.ipynb` — predicting survival for the same passenger input:
```
[Pclass=2, Sex='male', Age=31.0, SibSp=0, Parch=0, Fare=10.5, Embarked='S']
```

The raw input array is created identically. The crucial difference is what happens **next**.

In [6]:
test_input2 = np.array([2,'male',31.0,0,0,10.5,'S'],dtype=object).reshape(1,7)

### One Call — Entire Pipeline Executes Automatically

`pipe.predict(test_input2)` internally runs the full 5-step chain:

```
raw input (1×7 array)
    → trf1: impute Age and Embarked (no-op here, both provided)
    → trf2: OHE Sex ('male' → [0,1]) and Embarked ('S' → [0,0,1])  ← correct columns automatically
    → trf3: MinMaxScale all 10 columns
    → trf4: SelectKBest — keep the 8 most informative features
    → DecisionTree: predict
→ output: array([0])
```

**There is no way to pass the wrong column to the wrong encoder.** The pipeline knows which column index maps to which transformer — this was defined once during construction in `UsingPipeline1.ipynb` and is baked into `pipe.pkl`.

### About the Warnings

```
UserWarning: X does not have valid feature names, but SimpleImputer was fitted with feature names
```
These warnings appear because `pipe` was fitted on a pandas DataFrame (which has column names), but the input here is a raw numpy array (no column names). This is harmless — the pipeline still processes everything correctly by position. To suppress: pass a DataFrame with matching column names instead of a numpy array.

### Output: `array([0])` — Predicted: Did Not Survive

Note this differs from the `WithoutPipeline2` result of `[1]`. The pipeline result is **more trustworthy** because:
1. All column → transformer mappings are correct (no silent bug like the index-1-vs-index-6 mistake)
2. The pipeline includes additional steps (feature selection, better-tuned tree depth) not present in the manual version

---

## Summary — Without Pipeline vs With Pipeline

| Aspect | Without Pipeline | With Pipeline |
|--------|-----------------|---------------|
| **Training code** | 17+ cells of manual steps | `pipe.fit(X_train, y_train)` |
| **Saved files** | 3 `.pkl` files | 1 `.pkl` file |
| **Inference code** | 10+ cells, manual column indexing | `pipe.predict(raw_input)` |
| **Data leakage risk** | Must manually remember to only fit on train | Structurally impossible |
| **Cross-validation** | Would require re-running all transforms per fold | `cross_val_score(pipe, ...)` — one line |
| **Hyperparameter tuning** | Infeasible manually | `GridSearchCV(pipe, params)` — clean |
| **Silent bugs** | High risk (wrong index, wrong order) | No risk — mapping is structural |
| **Deployment** | Load 3 files, run 10 steps | Load 1 file, call predict |

In [7]:
pipe.predict(test_input2)

C:\...\validation.py:2749: UserWarning: X does not have valid feature names, but SimpleImputer was fitted with feature names
  warnings.warn(
C:\...\validation.py:2749: UserWarning: X does not have valid feature names, but SimpleImputer was fitted with feature names
  warnings.warn(


array([0])